In [1]:
import os

# This guide can only be run with the torch backend.
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' # Only errors are logged
os.environ['TF_GPU_ALLOCATOR'] ='cuda_malloc_async'

import keras
from keras import layers
import numpy as np
import os
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf

from keras import models
from IPython import display
import scipy.io.wavfile

import pandas as pd
from scipy.signal import resample

keras.utils.set_random_seed(41)

I0000 00:00:1774305958.086214   94804 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774305960.331240   94804 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
BASE_DATA_DIR = "./datasets/"
BATCH_SIZE = 64
NUM_CLASSES = 8
EPOCHS = 200
SAMPLE_RATE = 48000

In [3]:
train_ds, val_ds = tf.keras.utils.audio_dataset_from_directory(
    directory='datasets',
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    seed=41,
    # output_sequence_length=None,
    label_mode = 'int',
    subset='both')

label_names = np.array(train_ds.class_names)
print()
print("label names:", label_names)


Found 4240 files belonging to 8 classes.
Using 3392 files for training.
Using 848 files for validation.

label names: ['angry' 'calm' 'disgust' 'fear' 'happy' 'neutral' 'sad' 'surprised']


In [4]:
test_ds = val_ds.shard(num_shards=2, index=0)
val_ds = val_ds.shard(num_shards=2, index=1)


In [5]:
stftlayer = layers.STFTSpectrogram(
            mode="log",
            frame_length=SAMPLE_RATE * 40 // 1000,
            frame_step=SAMPLE_RATE * 15 // 1000,
            trainable=False,
        )

In [6]:
for i in train_ds.take(1):
    print(i[0].shape)
    stftl = stftlayer(i[0])

(64, 70470, 1)


In [7]:
stftl.shape

TensorShape([64, 96, 1025])

In [8]:
model1d = keras.Sequential(
    [
        layers.InputLayer((None, 1)),
        layers.STFTSpectrogram(
            mode="log",
            # frame_length=SAMPLE_RATE * 40 // 1000,
            # frame_step=SAMPLE_RATE * 15 // 1000,
            trainable=False,
        ),
        layers.Conv1D(64, 64, activation="relu"),
        layers.Conv1D(128, 16, activation="relu"),
        layers.LayerNormalization(),
        layers.MaxPooling1D(4),
        layers.Conv1D(128, 8, activation="relu"),
        layers.Conv1D(256, 8, activation="relu"),
        layers.Conv1D(512, 4, activation="relu"),
        layers.LayerNormalization(),
        layers.Dropout(0.5),
        layers.GlobalMaxPooling1D(),
        layers.Dense(256, activation="relu"),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ],
    name="model_1d_non_trainble_stft",
)
model1d.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model1d.summary()

Model: "model_1d_non_trainble_stft"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ stft_spectrogram_1              │ (None, None, 129)      │        66,048 │
│ (STFTSpectrogram)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, None, 64)       │       528,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, None, 128)      │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization             │ (None, None, 128)      │           256 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, None, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, None, 128)      │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, None, 256)      │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, None, 512)      │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_1           │ (None, None, 512)      │         1,024 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, None, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 512)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │         2,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,844,552 (7.04 MB)

 Trainable params: 1,778,504 (6.78 MB)

 Non-trainable params: 66,048 (258.00 KB)

In [ ]:
history_model1d = model1d.fit(
    train_ds,
    batch_size=BATCH_SIZE,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EPOCHS,
            restore_best_weights=True,
        )
    ],
)

Epoch 1/200
53/53 ━━━━━━━━━━━━━━━━━━━━ 208s 4s/step - accuracy: 0.1368 - loss: 2.4112 - val_accuracy: 0.1175 - val_loss: 2.0851
Epoch 2/200
53/53 ━━━━━━━━━━━━━━━━━━━━ 15s 292ms/step - accuracy: 0.1574 - loss: 2.1060 - val_accuracy: 0.1350 - val_loss: 2.0785
Epoch 3/200
53/53 ━━━━━━━━━━━━━━━━━━━━ 6s 118ms/step - accuracy: 0.1822 - loss: 2.0397 - val_accuracy: 0.1525 - val_loss: 2.0522
Epoch 4/200
53/53 ━━━━━━━━━━━━━━━━━━━━ 6s 112ms/step - accuracy: 0.2134 - loss: 1.9724 - val_accuracy: 0.1425 - val_loss: 2.0440
Epoch 5/200
53/53 ━━━━━━━━━━━━━━━━━━━━ 12s 233ms/step - accuracy: 0.2535 - loss: 1.9141 - val_accuracy: 0.2075 - val_loss: 2.0236
Epoch 6/200
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.2916 - loss: 1.8468 - val_accuracy: 0.2125 - val_loss: 1.9958
Epoch 7/200
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.3228 - loss: 1.7746 - val_accuracy: 0.2625 - val_loss: 1.9256
Epoch 8/200
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.3753 - loss: 1.6633 - val_accuracy

: 